[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/BMED365-2026/blob/main/Lab5-Comp-Mod/MLX-Bio-Qwen/notebooks/03-compmod-fine-tuning.ipynb)

Arvid Lundervold, 2026-02-11

# BMED365: Lab 5 — Fine-Tuning Qwen 2.5 72B Instruct Locally

## A Practical Guide to LoRA/QLoRA Fine-Tuning on Apple Silicon

**Objective:** This notebook provides a **comprehensive, step-by-step guide** for fine-tuning the local Qwen 2.5 72B Instruct model for domain-specific tasks in computational medicine, neuroscience, and biomedical physics. We use **LoRA** (Low-Rank Adaptation) or **QLoRA** (quantized LoRA) via Apple's MLX framework to adapt the model efficiently on a Mac without full parameter updates.

**Target setup:**
- **Platform:** Apple Silicon Mac (M1–M4, preferably 64GB+ unified memory; 128GB recommended for 72B)
- **Environment:** `mlx-bio` Conda environment
- **Model:** `mlx-community/Qwen2.5-72B-Instruct-4bit` (QLoRA fine-tuning)

### Notes & Prerequisites

- **Environment:** Create the `mlx-bio` Conda env from `environment-mlx-bio.yml`. It includes `mlx-lm[train]` for LoRA training.
- **Working directory:** Run this notebook from `MLX-Bio-Qwen/notebooks/` so `lora_data/` and `adapters/` are created in the expected locations.
- **Kernel restart:** If Step 1 installs `mlx-lm[train]`, **restart the kernel** and re-run from the top before fine-tuning.
- **Original model unchanged:** LoRA creates separate adapter files; the base model (used in notebook 02) is never modified.
- **Generated files:** `adapters/` (~100MB–1GB) and `fused_model/` (~36–45GB if fused) are in `.gitignore` and should not be committed.
- **Runtime:** Fine-tuning 72B for 100 iters may take roughly 1–3 hours on an M4 Max (128GB).

---

## 1. Rationale for Fine-Tuning

### 1.1 Why Fine-Tune at All?

Pre-trained large language models (e.g., Qwen 2.5 72B) are trained on enormous general-purpose corpora and excel at broad tasks. However, for **domain-specific applications** — such as computational medicine, neurodynamics, or medical physics — they may:

- **Use suboptimal terminology** — favoring generic phrasing over domain-standard jargon
- **Prioritize incorrect models** — e.g., choosing Euler over Euler-Maruyama for SDEs, or standard ODE solvers for stochastic systems
- **Generate plausible but incorrect formulas** — especially in niche areas (e.g., Bloch equations, Hodgkin-Huxley kinetics)
- **Lack task-specific format preferences** — e.g., always showing equations in LaTeX, using vectorized NumPy, or including units in plots

**Fine-tuning** adapts the model's weights to your specific domain and task, so it learns the desired behavior from **few-shot examples** rather than relying on prompting alone.

### 1.2 When Is Fine-Tuning Worth It?

| Scenario | Recommendation |
|----------|----------------|
| You have &lt; 50 high-quality examples | Prefer **prompt engineering** or **RAG** (retrieval-augmented generation) |
| You have 100–10,000 curated examples | **LoRA/QLoRA fine-tuning** is ideal |
| You need consistent output format/style | Fine-tuning can enforce structure (e.g., always LaTeX, always vectorized code) |
| You have sensitive data (HIPAA, genomics) | Local fine-tuning keeps data on your machine |

### 1.3 LoRA vs. Full Fine-Tuning

| Approach | Parameters Updated | Memory | Use Case |
|----------|--------------------|--------|----------|
| **Full fine-tuning** | All 72B | Very high (multi-GPU) | Rarely feasible on consumer hardware |
| **LoRA** | Small adapter matrices (~0.1–1% of params) | Moderate | Standard choice for large models |
| **QLoRA** | Same as LoRA, but base model is quantized (4-bit) | Lower | **Recommended for 72B on Apple Silicon** |

**LoRA** (Low-Rank Adaptation) injects trainable low-rank matrices into selected layers. Instead of updating all 72B parameters, we train only these small adapters — typically a few hundred MB — and apply them at inference time. **QLoRA** further reduces memory by keeping the base model in 4-bit quantization during training.

---

## 2. Overview of the Fine-Tuning Pipeline

The process has five main steps:

1. **Install training dependencies** — `mlx-lm[train]` extends `mlx-lm` with LoRA/QLoRA support.
2. **Prepare training data** — JSONL format (`train.jsonl`, `valid.jsonl` required; `test.jsonl` optional) with chat or completion examples.
3. **Run fine-tuning** — Invoke `mlx_lm.lora` with the model path, data path, and hyperparameters.
4. **Evaluate** (optional) — Compute test perplexity or sample outputs.
5. **Inference with adapters** — Load the base model + adapters, or fuse them into a single model.

We document each step in detail below.

---

## 3. Step 1: Environment Check and Training Dependencies

**Rationale:** The base `mlx-lm` package supports inference only. The `[train]` extra installs dependencies for LoRA training (e.g., optimizer, gradient computation). Without it, `mlx_lm.lora` will not be available.

**What we do:** Verify MLX is available and install `mlx-lm[train]` if needed.

In [1]:
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
HAS_MLX = False
try:
    import mlx.core as mx
    HAS_MLX = True
except ImportError:
    pass

if HAS_MLX:
    print(f"✅ MLX Device: {mx.default_device()}")
else:
    print("⚠️ MLX not available. Fine-tuning requires Apple Silicon and the mlx-bio environment.")

# Install training extras if not already present
try:
    from mlx_lm import lora
    print("✅ mlx-lm[train] already installed.")
except ImportError:
    print("Installing mlx-lm[train]...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx-lm[train]"])

✅ MLX Device: Device(gpu, 0)
✅ mlx-lm[train] already installed.


---

## 4. Step 2: Prepare Training Data

**Rationale:** Fine-tuning requires **supervised examples** in a format the trainer expects. Each example is a (prompt, completion) pair or a multi-turn conversation. The model learns to mimic the style, terminology, and correctness of your examples.

### 4.1 Supported Data Formats

| Format | Key structure | Use case |
|--------|----------------|----------|
| **chat** | `{"messages": [{"role": "system/user/assistant", "content": "..."}]}` | Multi-turn, system prompts |
| **completions** | `{"prompt": "...", "completion": "..."}` | Single-turn Q&A |
| **text** | `{"text": "..."}` | Unstructured continuation |

For computational medicine Q&A, we use the **chat** format with a system prompt that encodes domain expertise.

### 4.2 Data Quality Guidelines

- **Consistency:** Use the same conventions (LaTeX $...$ vs $$...$$, units, variable names) across examples.
- **Correctness:** Ensure mathematical and scientific content is accurate.
- **Format:** One JSON object per line; no line breaks within a single example.

### 4.3 Create a Sample Dataset

Below we create a minimal `train.jsonl` and `valid.jsonl` with a few computational medicine examples. **mlx_lm.lora requires a validation set** for fine-tuning. In practice, you would expand to hundreds or thousands of curated examples.

In [2]:
from pathlib import Path

# Directory for fine-tuning data
DATA_DIR = Path("lora_data")
DATA_DIR.mkdir(exist_ok=True)
print(f"Data directory: {DATA_DIR.absolute()}")

Data directory: /Users/arvid/GitHub/BMED365-2026/Lab5-Comp-Mod/MLX-Bio-Qwen/notebooks/lora_data


In [4]:
import json

SYSTEM_PROMPT = (
    "You are an expert Professor of Computational Medicine and Biomedical Physics. "
    "Your goal is to translate biological problems into precise, high-performance computational models. "
    "Always write governing equations in LaTeX ($...$ inline, $$...$$ display). "
    "Use vectorized NumPy; for SDEs use Euler-Maruyama, never standard ODE solvers for noise."
)

# Example training examples (chat format)
EXAMPLES = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "How do I model mean-reverting gene expression?"},
            {
                "role": "assistant",
                "content": (
                    "Use the Ornstein-Uhlenbeck SDE: $dX_t = \\theta(\\mu - X_t)dt + \\sigma dW_t$. "
                    "Here $\\theta$ is reversion speed, $\\mu$ is the mean, $\\sigma$ is volatility. "
                    "Implement with Euler-Maruyama: $X_{t+1} = X_t + \\theta(\\mu - X_t)\\Delta t + \\sigma \\sqrt{\\Delta t} \\cdot z$ with $z \\sim N(0,1)$."
                ),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "What equations govern tumor growth?"},
            {
                "role": "assistant",
                "content": (
                    "Gompertzian: $\\frac{dV}{dt} = aV \\ln\\frac{V_{\\infty}}{V}$. "
                    "Logistic: $\\frac{dV}{dt} = rV(1 - V/K)$. "
                    "Von Bertalanffy: $\\frac{dV}{dt} = \\eta V^{2/3} - \\lambda V$. "
                    "Use scipy.integrate.solve_ivp for deterministic integration."
                ),
            },
        ]
    },
    {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Write the Bloch equations for MRI."},
            {
                "role": "assistant",
                "content": (
                    "The Bloch equations: $\\frac{d\\mathbf{M}}{dt} = \\gamma \\mathbf{M} \\times \\mathbf{B} - \\frac{M_x\\mathbf{\\hat{x}} + M_y\\mathbf{\\hat{y}}}{T_2} - \\frac{(M_z - M_0)\\mathbf{\\hat{z}}}{T_1}$. "
                    "Use rotation matrices for RF pulses and relaxation; vectorize over voxels with NumPy."
                ),
            },
        ]
    },
]

# Split into train and validation (mlx_lm.lora requires valid.jsonl)
TRAIN_EXAMPLES = EXAMPLES[:2]
VALID_EXAMPLES = EXAMPLES[2:3]  # Hold out 1 for validation

train_path = DATA_DIR / "train.jsonl"
with open(train_path, "w") as f:
    for ex in TRAIN_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

valid_path = DATA_DIR / "valid.jsonl"
with open(valid_path, "w") as f:
    for ex in VALID_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Wrote {len(TRAIN_EXAMPLES)} examples to {train_path}")
print(f"Wrote {len(VALID_EXAMPLES)} examples to {valid_path}")
print("First example (user message):", EXAMPLES[0]["messages"][1]["content"][:80] + "...")

Wrote 2 examples to lora_data/train.jsonl
Wrote 1 examples to lora_data/valid.jsonl
First example (user message): How do I model mean-reverting gene expression?...


---

## 5. Step 3: Run Fine-Tuning

**Rationale:** The `mlx_lm.lora` command loads the model, attaches LoRA adapters to selected layers, and performs gradient updates on a batch of examples. For a **quantized** model (e.g., 4-bit), it automatically uses **QLoRA**.

### 5.1 Key Hyperparameters

| Parameter | Role | Suggestion for 72B |
|-----------|------|--------------------|
| `--iters` | Number of gradient steps | 100–1000 (more data ⇒ more iters) |
| `--batch-size` | Examples per step | 1–2 (memory-limited for 72B) |
| `--num-layers` | Layers with LoRA adapters | 4–8 (fewer = less memory) |
| `--lora-layers` | Same as num-layers | Use if available |
| `--grad-checkpoint` | Trade compute for memory | Recommended for 72B |
| `--learning-rate` | Step size | 1e-5 to 1e-4 |
| `--adapter-path` | Where to save adapters | `adapters/` (default) |

### 5.2 Memory Considerations (72B on Apple Silicon)

- **4-bit quantized base:** ~36–45 GB
- **LoRA adapters + gradients:** Additional 10–30 GB depending on batch size and layers
- **Recommendation:** 128 GB unified memory for comfortable 72B QLoRA; 64 GB may work with `batch-size 1`, `num-layers 4`, `--grad-checkpoint`

For machines with less memory, use a smaller model (e.g., `mlx-community/Qwen2.5-7B-Instruct`) by changing `MODEL_ID` below.

In [5]:
# Configuration for fine-tuning
MODEL_ID = "mlx-community/Qwen2.5-72B-Instruct-4bit"
ADAPTER_PATH = Path("adapters")
ADAPTER_PATH.mkdir(exist_ok=True)

# Conservative settings for 72B on 128GB (or 64GB with grad-checkpoint)
TRAIN_ARGS = [
    "--model", MODEL_ID,
    "--train",
    "--data", str(DATA_DIR),
    "--iters", "100",           # Use 100 for demo; increase for real training
    "--batch-size", "1",        # Reduce memory
    "--num-layers", "4",        # Fewer LoRA layers = less memory
    "--adapter-path", str(ADAPTER_PATH),
    "--grad-checkpoint",         # Trade compute for memory
    "--learning-rate", "1e-5",
]

print("Fine-tuning command (conceptual):")
print("mlx_lm.lora", " ".join(TRAIN_ARGS))

Fine-tuning command (conceptual):
mlx_lm.lora --model mlx-community/Qwen2.5-72B-Instruct-4bit --train --data lora_data --iters 100 --batch-size 1 --num-layers 4 --adapter-path adapters --grad-checkpoint --learning-rate 1e-5


In [5]:
%%time
if HAS_MLX:
    # Run fine-tuning via subprocess (mlx_lm.lora is a CLI entry point)
    cmd = [sys.executable, "-m", "mlx_lm.lora"] + TRAIN_ARGS
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        print("⚠️ Fine-tuning exited with non-zero code.")
        print("   If out-of-memory, try: smaller batch-size, fewer num-layers, or Qwen2.5-7B-Instruct.")
else:
    print("⏭️ Skipping fine-tuning (MLX not available).")

Running: /opt/anaconda3/envs/mlx-bio/bin/python -m mlx_lm.lora --model mlx-community/Qwen2.5-72B-Instruct-4bit --train --data lora_data --iters 100 --batch-size 1 --num-layers 4 --adapter-path adapters --grad-checkpoint --learning-rate 1e-5
Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading pretrained model


Fetching 16 files: 100%|██████████| 16/16 [00:00<00:00, 34117.37it/s]


Loading datasets
Training
Trainable parameters: 0.007% (5.263M/72706.204M)
Starting training..., iters: 100


mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.
Calculating loss...: 100%|██████████| 1/1 [00:04<00:00,  4.33s/it]


Iter 1: Val loss 2.755, Val took 4.331s
Iter 10: Train loss 2.370, Learning Rate 1.000e-05, It/sec 0.352, Tokens/sec 66.915, Trained Tokens 1900, Peak mem 41.856 GB
Iter 20: Train loss 1.011, Learning Rate 1.000e-05, It/sec 0.350, Tokens/sec 66.464, Trained Tokens 3800, Peak mem 41.856 GB
Iter 30: Train loss 0.297, Learning Rate 1.000e-05, It/sec 0.362, Tokens/sec 68.774, Trained Tokens 5700, Peak mem 41.856 GB
Iter 40: Train loss 0.039, Learning Rate 1.000e-05, It/sec 0.362, Tokens/sec 68.736, Trained Tokens 7600, Peak mem 41.856 GB
Iter 50: Train loss 0.006, Learning Rate 1.000e-05, It/sec 0.359, Tokens/sec 68.140, Trained Tokens 9500, Peak mem 41.856 GB
Iter 60: Train loss 0.005, Learning Rate 1.000e-05, It/sec 0.360, Tokens/sec 68.425, Trained Tokens 11400, Peak mem 41.856 GB
Iter 70: Train loss 0.004, Learning Rate 1.000e-05, It/sec 0.357, Tokens/sec 67.802, Trained Tokens 13300, Peak mem 41.856 GB
Iter 80: Train loss 0.004, Learning Rate 1.000e-05, It/sec 0.362, Tokens/sec 68.704

Calculating loss...: 100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


Iter 100: Val loss 0.867, Val took 2.820s
Iter 100: Train loss 0.004, Learning Rate 1.000e-05, It/sec 0.358, Tokens/sec 68.028, Trained Tokens 19000, Peak mem 41.856 GB
Iter 100: Saved adapter weights to adapters/adapters.safetensors and adapters/0000100_adapters.safetensors.
Saved final weights to adapters/adapters.safetensors.
CPU times: user 11.2 ms, sys: 12.5 ms, total: 23.7 ms
Wall time: 4min 55s


### 5.3 What Happens During Training

1. **Load model:** The quantized base model is loaded into unified memory.
2. **Attach LoRA:** Low-rank matrices are added to the specified layers (default: attention layers).
3. **Data loop:** Batches from `train.jsonl` are tokenized and fed through the model.
4. **Loss:** Cross-entropy loss is computed (optionally with `--mask-prompt` to ignore the prompt tokens).
5. **Backprop:** Gradients flow only through LoRA parameters.
6. **Save:** Adapter weights are written to `adapter-path`.

---

## 6. Step 4: Evaluate (Optional)

**Rationale:** After training, you can compute **perplexity** on a held-out `test.jsonl` to measure how well the model fits unseen data. Lower perplexity generally indicates better fit.

**Note:** You need a `test.jsonl` in the data directory. We skip evaluation here if the file is absent.

In [8]:
test_path = DATA_DIR / "test.jsonl"
if test_path.exists() and ADAPTER_PATH.exists() and HAS_MLX:
    cmd = [
        sys.executable, "-m", "mlx_lm.lora",
        "--model", MODEL_ID,
        "--adapter-path", str(ADAPTER_PATH),
        "--data", str(DATA_DIR),
        "--test",
    ]
    subprocess.run(cmd)
else:
    print("Skipping evaluation: test.jsonl or adapters/ not found.")

Skipping evaluation: test.jsonl or adapters/ not found.


---

## 7. Step 5: Inference with Adapters

**Rationale:** Trained adapters are **additive** — they modify the base model's behavior without changing the base weights. At inference, you load the base model **and** the adapters; `mlx_lm` applies them automatically.

### 7.1 Generate with Adapters

Use `load` with `adapter_path` to combine base + adapters, then `generate` as usual.

In [6]:
if HAS_MLX and ADAPTER_PATH.exists():
    from mlx_lm import load, generate
    from mlx_lm.sample_utils import make_sampler

    print("Loading base model + adapters...")
    model, tokenizer = load(MODEL_ID, adapter_path=str(ADAPTER_PATH))
    print("✅ Ready for inference with fine-tuned adapters.")

    # Example query
    prompt = "How do I model tumor growth with the Gompertz equation?"
    messages = [{"role": "user", "content": prompt}]
    prompt_formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    response = generate(
        model, tokenizer, prompt=prompt_formatted, max_tokens=256,
        sampler=make_sampler(temp=0.3)
    )
    print("\n--- Generated response ---\n")
    print(response)
else:
    print("⏭️ Skipping inference (MLX or adapters not available).")

Loading base model + adapters...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Ready for inference with fine-tuned adapters.

--- Generated response ---

Use of the Gompertzian model is a common and effective way to model tumor growth. The Gompertzian model is particularly useful for biological systems because it captures the decelerating growth rate often observed in biological systems. Here’s how you can model tumor growth with the Gompertz equation:

### Gompertzian Equation
The Gompertzian model is given by the following differential equation:

\[
\frac{dV}{dt} = aV \ln\frac{V_{\infty}}{V}
\]

where:
- \( V \) is the tumor volume at time \( t \).
- \( V_{\infty} \) is the maximum tumor volume (carrying capacity).
- \( a \) is a growth rate parameter.

### Integration of the Gompertzian Equation
To find the tumor volume as a function of time, integrate the Gompertzian equation. The integration yields:

\[
V(t) = V_{\infty} e^{-e^{aV_{\infty}e^{-at} + \ln(-\ln(V(0)/V_{\infty}))}}
\]

Here:
- \( V(0) \) is the initial tumor volume at time \(


### 7.2 Fuse Adapters (Optional)

**Rationale:** If you want a **single model file** without separate adapter loading, you can **fuse** the LoRA weights into the base model. This produces a standalone model that already incorporates the fine-tuned behavior.

**Command:**
```bash
mlx_lm.fuse --model mlx-community/Qwen2.5-72B-Instruct-4bit --adapter-path adapters/
```

This writes the fused model to `fused_model/` by default. You can then load it like any MLX model without specifying `adapter_path`.

In [7]:
if HAS_MLX and ADAPTER_PATH.exists():
    FUSED_PATH = Path("fused_model")
    cmd = [
        sys.executable, "-m", "mlx_lm.fuse",
        "--model", MODEL_ID,
        "--adapter-path", str(ADAPTER_PATH),
    ]
    # Uncomment to run (creates fused_model/):
    # subprocess.run(cmd)
    print("To fuse adapters, run in terminal:")
    print(" ".join(cmd))
else:
    print("Adapters not found; skipping fuse.")

To fuse adapters, run in terminal:
/opt/anaconda3/envs/mlx-bio/bin/python -m mlx_lm.fuse --model mlx-community/Qwen2.5-72B-Instruct-4bit --adapter-path adapters


---

## 8. Summary and Next Steps

| Step | What we did |
|------|-------------|
| 1 | Installed `mlx-lm[train]` for LoRA/QLoRA support |
| 2 | Created `train.jsonl` in chat format with domain examples |
| 3 | Ran `mlx_lm.lora` with conservative memory settings for 72B |
| 4 | (Optional) Evaluated on test set |
| 5 | Loaded model + adapters for inference; optionally fused |

**To scale up:**
- Add hundreds or thousands of high-quality examples to `train.jsonl`
- Increase `--iters` proportionally
- Create `valid.jsonl` for validation loss monitoring
- Consider `--mask-prompt` if you only want to optimize completion tokens

**References:**
- [MLX-LM LoRA documentation](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md)
- [LoRA paper (Hu et al.)](https://arxiv.org/abs/2106.09685)
- [QLoRA paper](https://arxiv.org/abs/2305.14314)